In [4]:
# import numpy as np
# import pandas as pd
# import neurokit2 as nk


# class ECGMetricCalculator:
#     """
#     Calcula um conjunto expandido de métricas (HRV e Morfológicas) para
#     validação contra o CSV clínico.
#     """
#     def __init__(self, cleaned_signal, sampling_rate=200, debug=True):
#         self.sampling_rate = sampling_rate
#         self.debug = debug
        
#         if self.debug:
#             print(f"📊 Sinal recebido: {len(cleaned_signal)} amostras")
        
#         self.ecg_limpo = nk.ecg_clean(cleaned_signal, sampling_rate=200, method='elgendi2010')
        
#         if self.debug:
#             print(f"✅ Sinal limpo: {len(self.ecg_limpo)} amostras")
        
#         self.epochs = self._create_epochs()
        
#         if self.debug:
#             print(f"📦 Épocas criadas: {len(self.epochs)} épocas")
#             print(f"   Nomes das épocas: {list(self.epochs.keys())}")

#     def _create_epochs(self):
#         epoch_duration_seconds = 120
#         epoch_length_samples = epoch_duration_seconds * self.sampling_rate
#         num_epochs = len(self.ecg_limpo) // epoch_length_samples
        
#         if self.debug:
#             print(f"   Duração da época: {epoch_duration_seconds}s ({epoch_length_samples} amostras)")
#             print(f"   Número de épocas possíveis: {num_epochs}")
        
#         events = [i * epoch_length_samples for i in range(num_epochs)]
        
#         return nk.epochs_create(
#             self.ecg_limpo, 
#             events=events, 
#             sampling_rate=self.sampling_rate, 
#             epochs_start=0, 
#             epochs_end=epoch_duration_seconds
#         )

#     def calculate_metrics_for_epochs(self):
#         all_metrics = []
#         epochs_processadas = 0
#         epochs_com_erro = 0
        
#         if self.debug:
#             print("\n" + "="*60)
#             print("🔍 INICIANDO PROCESSAMENTO DAS ÉPOCAS")
#             print("="*60)
        
#         for epoch_name, epoch_df in self.epochs.items():
#             if self.debug:
#                 print(f"\n📌 Processando época '{epoch_name}'...")
            
#             signal = epoch_df["Signal"].values
            
#             if self.debug:
#                 print(f"   Tamanho do sinal da época: {len(signal)} amostras")

#             try:
#                 # Detecção de picos R
#                 peaks, info = nk.ecg_peaks(signal, sampling_rate=self.sampling_rate)
#                 num_picos = len(info['ECG_R_Peaks'])
                
#                 if self.debug:
#                     print(f"   ✓ Picos R encontrados: {num_picos}")
                
#                 # Verifica se há picos suficientes
#                 if num_picos < 5:
#                     if self.debug:
#                         print(f"   ⚠️  PULANDO: Poucos picos R ({num_picos} < 5)")
#                     epochs_com_erro += 1
#                     continue

#                 # ============================================================
#                 # MÉTODO 1: nk.hrv_time() - Métricas de HRV no domínio do tempo
#                 # ============================================================
#                 hrv_metrics_v1 = nk.hrv_time(peaks, sampling_rate=self.sampling_rate, show=False)
#                 #hrv_metrics_v1 = nk.hrv(peaks, sampling_rate=self.sampling_rate, show=False)
                
#                 hrv_mean_nn_v1 = hrv_metrics_v1['HRV_MeanNN'].values[0]
#                 hrv_min_nn_v1 = hrv_metrics_v1['HRV_MinNN'].values[0] if 'HRV_MinNN' in hrv_metrics_v1.columns else np.nan
#                 hrv_max_nn_v1 = hrv_metrics_v1['HRV_MaxNN'].values[0] if 'HRV_MaxNN' in hrv_metrics_v1.columns else np.nan
                
#                 # Calcula manualmente se não estiver disponível
#                 if pd.isna(hrv_min_nn_v1) or pd.isna(hrv_max_nn_v1):
#                     rr_intervals = np.diff(info["ECG_R_Peaks"]) / self.sampling_rate * 1000  # em ms
#                     hrv_min_nn_v1 = np.min(rr_intervals)
#                     hrv_max_nn_v1 = np.max(rr_intervals)
                
#                 rr_range_v1 = hrv_max_nn_v1 - hrv_min_nn_v1
#                 hrv_sdnn_v1 = hrv_metrics_v1['HRV_SDNN'].values[0]
#                 hrv_rmssd_v1 = hrv_metrics_v1['HRV_RMSSD'].values[0]
#                 hrv_pnn50_v1 = hrv_metrics_v1['HRV_pNN50'].values[0]
                
#                 if self.debug:
#                     print(f"   ✓ Método 1 (nk.hrv_time) processado")
                
#                 # ============================================================
#                 # MÉTODO 2: nk.ecg_intervalrelated - COM FALLBACK ROBUSTO
#                 # ============================================================
#                 try:
#                     hrv_metrics_v2 = nk.ecg_intervalrelated(signal, sampling_rate=self.sampling_rate)
                    
#                     hrv_mean_nn_v2 = hrv_metrics_v2['HRV_MeanNN'].values[0]
#                     hrv_min_nn_v2 = hrv_metrics_v2['HRV_MinNN'].values[0] if 'HRV_MinNN' in hrv_metrics_v2.columns else hrv_min_nn_v1
#                     hrv_max_nn_v2 = hrv_metrics_v2['HRV_MaxNN'].values[0] if 'HRV_MaxNN' in hrv_metrics_v2.columns else hrv_max_nn_v1
#                     rr_range_v2 = hrv_max_nn_v2 - hrv_min_nn_v2
#                     bradycardia_v2 = int(hrv_metrics_v2['ECG_Rate_Mean'].values[0] < 60)
#                     hrv_sdnn_v2 = hrv_metrics_v2['HRV_SDNN'].values[0]
#                     hrv_rmssd_v2 = hrv_metrics_v2['HRV_RMSSD'].values[0]
#                     hrv_pnn50_v2 = hrv_metrics_v2['HRV_pNN50'].values[0]
                    
#                     if self.debug:
#                         print(f"   ✓ Método 2 (nk.ecg_intervalrelated) processado")
                    
#                 except Exception as e_v2:
#                     # Fallback: usa valores do método 1
#                     if self.debug:
#                         print(f"   ⚠️  Método 2 falhou, usando Método 1 como fallback")
                    
#                     hrv_mean_nn_v2 = hrv_mean_nn_v1
#                     hrv_min_nn_v2 = hrv_min_nn_v1
#                     hrv_max_nn_v2 = hrv_max_nn_v1
#                     rr_range_v2 = rr_range_v1
#                     hrv_sdnn_v2 = hrv_sdnn_v1
#                     hrv_rmssd_v2 = hrv_rmssd_v1
#                     hrv_pnn50_v2 = hrv_pnn50_v1
                    
#                     # Calcula bradycardia manualmente
#                     rr_intervals_ms = np.diff(info["ECG_R_Peaks"]) / self.sampling_rate * 1000
#                     heart_rate = 60000 / np.mean(rr_intervals_ms) if len(rr_intervals_ms) > 0 else 60
#                     bradycardia_v2 = int(heart_rate < 60)
                
#                 # ============================================================
#                 # MÉTRICAS MORFOLÓGICAS (QRS/QT)
#                 # ============================================================
#                 try:
#                     delineate, _ = nk.ecg_delineate(
#                         signal, 
#                         rpeaks=info["ECG_R_Peaks"], 
#                         sampling_rate=self.sampling_rate, 
#                         method="dwt"
#                     )
                    
#                     # QRS Duration
#                     q_peaks = delineate[delineate["ECG_Q_Peaks"] == 1].index.to_numpy()
#                     s_peaks = delineate[delineate["ECG_S_Peaks"] == 1].index.to_numpy()
                    
#                     if len(q_peaks) == 0 or len(s_peaks) == 0:
#                         qrs_duration_mean = np.nan
#                         if self.debug:
#                             print(f"   ⚠️  QRS: Q_peaks={len(q_peaks)}, S_peaks={len(s_peaks)}")
#                     else:
#                         min_len_qs = min(len(q_peaks), len(s_peaks))
#                         pares_qs = [(q, s) for q, s in zip(q_peaks[:min_len_qs], s_peaks[:min_len_qs]) if s > q]
#                         qrs_duration_ms = [(s - q) / self.sampling_rate * 1000 for q, s in pares_qs]
#                         qrs_duration_mean = np.nanmean(qrs_duration_ms) if len(qrs_duration_ms) > 0 else np.nan

#                     # QT Interval (corrigido por Bazett)
#                     t_offsets = delineate[delineate["ECG_T_Offsets"] == 1].index.to_numpy()
                    
#                     if len(q_peaks) == 0 or len(t_offsets) == 0:
#                         qt_interval_mean = np.nan
#                         if self.debug:
#                             print(f"   ⚠️  QT: Q_peaks={len(q_peaks)}, T_offsets={len(t_offsets)}")
#                     else:
#                         min_len_qt = min(len(q_peaks), len(t_offsets))
#                         pares_qt = [(q, t) for q, t in zip(q_peaks[:min_len_qt], t_offsets[:min_len_qt]) if t > q]
#                         qt_interval_ms = [(t - q) / self.sampling_rate * 1000 for q, t in pares_qt]
#                         qt_interval_raw_mean = np.nanmean(qt_interval_ms) if len(qt_interval_ms) > 0 else np.nan
                        
#                         # Correção de Bazett: QTc = QT / sqrt(RR)
#                         rr_intervals_sec = np.diff(info["ECG_R_Peaks"]) / self.sampling_rate
#                         if len(rr_intervals_sec) > 0:
#                             rr_medio_sec = np.mean(rr_intervals_sec)
#                             if pd.notna(qt_interval_raw_mean) and rr_medio_sec > 0:
#                                 qt_interval_mean = qt_interval_raw_mean / np.sqrt(rr_medio_sec)
#                             else:
#                                 qt_interval_mean = np.nan
#                         else:
#                             qt_interval_mean = np.nan
                    
#                     if self.debug:
#                         print(f"   ✓ Métricas morfológicas processadas")
                        
#                 except Exception as e_morph:
#                     if self.debug:
#                         print(f"   ⚠️  Métricas morfológicas falharam")
#                     qrs_duration_mean = np.nan
#                     qt_interval_mean = np.nan

#                 # ============================================================
#                 # DICIONÁRIO FINAL
#                 # ============================================================
#                 metrics_dict = {
#                     'Epoca': epoch_name,
#                     # Método 1 (hrv_time)
#                     'HRV_MeanNN_v1': hrv_mean_nn_v1,
#                     'HRV_MinNN_v1': hrv_min_nn_v1,
#                     'HRV_MaxNN_v1': hrv_max_nn_v1,
#                     'RR_Range_v1': rr_range_v1,
#                     'HRV_SDNN_v1': hrv_sdnn_v1,
#                     'HRV_RMSSD_v1': hrv_rmssd_v1,
#                     'HRV_pNN50_v1': hrv_pnn50_v1,
                    
#                     # Método 2 (ecg_intervalrelated ou fallback)
#                     'HRV_MeanNN_v2': hrv_mean_nn_v2,
#                     'HRV_MinNN_v2': hrv_min_nn_v2,
#                     'HRV_MaxNN_v2': hrv_max_nn_v2,
#                     'RR_Range_v2': rr_range_v2,
#                     'Bradycardia_v2': bradycardia_v2,
#                     'HRV_SDNN_v2': hrv_sdnn_v2,
#                     'HRV_RMSSD_v2': hrv_rmssd_v2,
#                     'HRV_pNN50_v2': hrv_pnn50_v2,
                    
#                     # Morfológicas
#                     'QRS_Duration_Mean': qrs_duration_mean,
#                     'QT_Interval_Mean': qt_interval_mean,
#                 }
#                 all_metrics.append(metrics_dict)
#                 epochs_processadas += 1
                
#                 if self.debug:
#                     print(f"   ✅ Época processada com sucesso!")
                
#             except Exception as e:
#                 epochs_com_erro += 1
#                 if self.debug:
#                     print(f"   ❌ ERRO ao processar época '{epoch_name}': {e}")
#                 continue
        
#         if self.debug:
#             print("\n" + "="*60)
#             print(f"✅ Épocas processadas com sucesso: {epochs_processadas}")
#             print(f"❌ Épocas com erro/puladas: {epochs_com_erro}")
#             print(f"📊 Total de métricas calculadas: {len(all_metrics)}")
#             print("="*60 + "\n")
                
#         return pd.DataFrame(all_metrics)

In [ ]:
"""
Script de automação para processar múltiplos pacientes e comparar com dados clínicos.
"""
import numpy as np
import pandas as pd
from ecg_utils import get_lista_pacientes, load_ecg_segment
from pipeline_limpeza import ECGMetricCalculator, processar_multiplos_pacientes


# ============================================================
# 1. CARREGAR DADOS DE REFERÊNCIA CLÍNICOS
# ============================================================
print("📋 Carregando dados clínicos de referência...")

path_clinical_csv = r'D:\cox-models-sudden-death\01_Dataset\dados_csv_info_definitions\subject-info_formatado.csv'
df_clinical = pd.read_csv(path_clinical_csv, sep=';')

column_mapping = {
    'Patient ID': 'paciente_id',
    'Average RR (ms)': 'HRV_MeanNN',
    'minimum RR (ms)': 'HRV_MinNN',
    'maximum RR (ms)': 'HRV_MaxNN',
    'RR range (ms)': 'RR_Range',
    'Bradycardia': 'Bradycardia',
    'SDNN (ms)': 'HRV_SDNN',
    'RMSSD (ms)': 'HRV_RMSSD',
    'pNN50 (%)': 'HRV_pNN50',
    'QRS duration (ms)': 'QRS_Duration_Mean',
    'QT interval (ms)': 'QT_Interval_Mean'
}

df_references = df_clinical[column_mapping.keys()].copy()
df_references.rename(columns=column_mapping, inplace=True)
df_references.set_index('paciente_id', inplace=True)

print(f"✅ {len(df_references)} pacientes no CSV clínico\n")


# ============================================================
# 2. FILTRAR PACIENTES VÁLIDOS
# ============================================================
print("🔍 Filtrando pacientes válidos...")

path_filtro_pacientes_csv = r'D:\cox-models-sudden-death\02_Preprocessamento_filtro\resumo_dataset_ecg.csv'
df_filtro = pd.read_csv(path_filtro_pacientes_csv, sep=',')

df_lista_validos = df_filtro[
    (df_filtro['tem_X'] == 'Sim') &
    (df_filtro['tem_Y'] == 'Sim') & 
    (df_filtro['tem_Z'] == 'Sim')
]

pacientes_validos = list(df_lista_validos['paciente'].values)
pacientes_para_processar = get_lista_pacientes(incluir_apenas=pacientes_validos)

print(f"✅ {len(pacientes_para_processar)} pacientes válidos\n")


# ============================================================
# 3. PROCESSAR PACIENTES (MODO TESTE - 5 PACIENTES)
# ============================================================
print("🧪 MODO TESTE: Processando primeiros 5 pacientes...\n")

# Para teste rápido, processa apenas 5 pacientes
pacientes_teste = pacientes_para_processar[:2]

df_epocas, df_resumo = processar_multiplos_pacientes(
    pacientes_lista=pacientes_teste,
    canal='x',
    minutos_a_pular=60,
    duracao_em_minutos=10,  # 10 minutos = 5 épocas de 2 min
    sampling_rate=200,
    debug=False  # True para ver detalhes
)

print("\n📊 RESULTADOS:")
print(f"   Épocas processadas: {len(df_epocas)}")
print(f"   Pacientes processados: {len(df_resumo)}")


# ============================================================
# 4. COMPARAR COM DADOS CLÍNICOS
# ============================================================
print("\n" + "="*70)
print("📊 COMPARAÇÃO: Calculado vs Clínico")
print("="*70)

# Métricas principais para comparar
metricas_comparar = [
    'HRV_MeanNN', 'HRV_SDNN', 'HRV_RMSSD', 
    'QRS_Duration_Mean', 'QT_Interval_Mean'
]

resultados_comparacao = []

for _, row in df_resumo.iterrows():
    paciente_id = row['Paciente_ID']
    
    if paciente_id not in df_references.index:
        continue
    
    ref = df_references.loc[paciente_id]
    
    print(f"\n🔹 {paciente_id}:")
    
    for metrica in metricas_comparar:
        # Valor calculado (média das épocas)
        col_mean = f'{metrica}_mean'
        if col_mean not in row:
            continue
            
        calculado = row[col_mean]
        clinico = ref[metrica]
        
        if pd.notna(calculado) and pd.notna(clinico):
            diff = calculado - clinico
            diff_pct = (diff / clinico * 100) if clinico != 0 else 0
            
            print(f"   {metrica:20s}: Calc={calculado:7.2f}  Clin={clinico:7.2f}  Diff={diff:+7.2f} ({diff_pct:+.1f}%)")
            
            resultados_comparacao.append({
                'Paciente_ID': paciente_id,
                'Metrica': metrica,
                'Calculado': calculado,
                'Clinico': clinico,
                'Diferenca': diff,
                'Diferenca_Percentual': diff_pct
            })

# Cria DataFrame de comparação
df_comparacao = pd.DataFrame(resultados_comparacao)


# ============================================================
# 5. ESTATÍSTICAS GERAIS DA COMPARAÇÃO
# ============================================================
print("\n" + "="*70)
print("📈 ESTATÍSTICAS GERAIS DE ERRO")
print("="*70)

for metrica in metricas_comparar:
    df_metrica = df_comparacao[df_comparacao['Metrica'] == metrica]
    
    if len(df_metrica) == 0:
        continue
    
    mae = df_metrica['Diferenca'].abs().mean()
    rmse = np.sqrt((df_metrica['Diferenca'] ** 2).mean())
    bias = df_metrica['Diferenca'].mean()
    
    print(f"\n{metrica}:")
    print(f"   MAE (Erro Médio Absoluto): {mae:.2f}")
    print(f"   RMSE (Raiz do Erro Quadrático): {rmse:.2f}")
    print(f"   BIAS (Viés): {bias:+.2f}")


# ============================================================
# 6. SALVAR RESULTADOS
# ============================================================
print("\n" + "="*70)
print("💾 SALVANDO RESULTADOS...")
print("="*70)

# Salva épocas
output_path_epocas = r'D:\cox-models-sudden-death\Arquitetura\logs\metricas_calibracao\metricas_por_epoca.csv'
df_epocas.to_csv(output_path_epocas, index=False, sep=';')
print(f"✅ Épocas salvas em: {output_path_epocas}")

# Salva resumo
output_path_resumo = r'D:\cox-models-sudden-death\Arquitetura\logs\metricas_calibracao\metricas_resumo.csv'
df_resumo.to_csv(output_path_resumo, index=False, sep=';')
print(f"✅ Resumo salvo em: {output_path_resumo}")

# Salva comparação
output_path_comparacao = r'D:\cox-models-sudden-death\Arquitetura\logs\metricas_calibracao\comparacao_calculado_vs_clinico.csv'
df_comparacao.to_csv(output_path_comparacao, index=False, sep=';')
print(f"✅ Comparação salva em: {output_path_comparacao}")


# ============================================================
# 7. PROCESSAR TODOS OS PACIENTES (COMENTADO POR PADRÃO)
# ============================================================

# Descomente para processar TODOS os pacientes válidos
print("\n" + "="*70)
print("🚀 PROCESSANDO TODOS OS PACIENTES...")
print("="*70 + "\n")

df_epocas_all, df_resumo_all = processar_multiplos_pacientes(
    pacientes_lista=pacientes_para_processar,
    canal='x',
    minutos_a_pular=60,
    duracao_em_minutos=10,
    sampling_rate=200,
    debug=False
)

# Salvar resultados completos
df_epocas_all.to_csv(r'D:\cox-models-sudden-death\Arquitetura\logs\metricas_calibracao\metricas_por_epoca_completo.csv', 
                     index=False, sep=';')
df_resumo_all.to_csv(r'D:\cox-models-sudden-death\Arquitetura\logs\metricas_calibracao\metricas_resumo_completo.csv', 
                     index=False, sep=';')

print("\n✅ PROCESSAMENTO COMPLETO FINALIZADO!")

print("\n" + "="*70)
print("✅ SCRIPT FINALIZADO COM SUCESSO!")
print("="*70)

📋 Carregando dados clínicos de referência...
✅ 992 pacientes no CSV clínico

🔍 Filtrando pacientes válidos...
✓ Filtrado para 874 pacientes incluídos
✅ 874 pacientes válidos

🧪 MODO TESTE: Processando primeiros 5 pacientes...


🚀 PROCESSANDO 2 PACIENTES

[1/2] Processando P0001...
✅ Sinal do paciente P0001 (canal X) carregado com sucesso.
   ✅ 5 épocas processadas

[2/2] Processando P0002...
✅ Sinal do paciente P0002 (canal X) carregado com sucesso.
   ✅ 5 épocas processadas

✅ PROCESSAMENTO CONCLUÍDO
   Total de pacientes processados: 2
   Total de épocas processadas: 10


📊 RESULTADOS:
   Épocas processadas: 10
   Pacientes processados: 2

📊 COMPARAÇÃO: Calculado vs Clínico

🔹 P0001:
   HRV_MeanNN          : Calc= 538.78  Clin= 984.00  Diff=-445.22 (-45.2%)
   QRS_Duration_Mean   : Calc=  67.73  Clin= 132.00  Diff= -64.27 (-48.7%)
   QT_Interval_Mean    : Calc= 320.82  Clin= 448.00  Diff=-127.18 (-28.4%)

🔹 P0002:
   HRV_MeanNN          : Calc= 700.92  Clin= 682.00  Diff= +18.92 (+2.

In [7]:
import pandas as pd

from ecg_utils import get_lista_pacientes, load_ecg_segment  # biblioteca local


# --- PREPARAÇÃO DOS DADOS DE REFERÊNCIA ---
path_clinical_csv = r'D:\cox-models-sudden-death\01_Dataset\dados_csv_info_definitions\subject-info_formatado.csv'
df_clinical = pd.read_csv(path_clinical_csv, sep=';')

column_mapping = {
    'Patient ID': 'paciente_id', 'Average RR (ms)': 'HRV_MeanNN',
    'minimum RR (ms)': 'HRV_MinNN', 'maximum RR (ms)': 'HRV_MaxNN',
    'RR range (ms)': 'RR_Range', 'Bradycardia': 'Bradycardia',
    'SDNN (ms)': 'HRV_SDNN', 'RMSSD (ms)': 'HRV_RMSSD',
    'pNN50 (%)': 'HRV_pNN50', 'QRS duration (ms)': 'QRS_Duration_Mean',
    'QT interval (ms)': 'QT_Interval_Mean'
}
df_references = df_clinical[column_mapping.keys()].copy()
df_references.rename(columns=column_mapping, inplace=True)
df_references.set_index('paciente_id', inplace=True)

#-------- filtros dos pacientes ----------------

path_filtro_pacientes_csv = r'D:\cox-models-sudden-death\02_Preprocessamento_filtro\resumo_dataset_ecg.csv'
df_filtro = pd.read_csv(path_filtro_pacientes_csv, sep=',')

df_lista_validos = df_filtro[(df_filtro['tem_X'] == 'Sim') &
                             (df_filtro['tem_Y'] == 'Sim') & 
                             (df_filtro['tem_Z'] == 'Sim')]
    
pacientes_validos = list(df_lista_validos['paciente'].values)

pacientes_para_processar = get_lista_pacientes(incluir_apenas=pacientes_validos)


sinal = load_ecg_segment('P0002', canal='x', minutos_a_pular=60, duracao_em_minutos=10, sampling_rate=200)

#ecg_limpo = nk.ecg_clean(sinal, sampling_rate=200, method='elgendi2010')
dados = ECGMetricCalculator(sinal, sampling_rate=200) # ,debug=False
teste = dados.calculate_metrics_for_epochs()

✓ Filtrado para 874 pacientes incluídos
✅ Sinal do paciente P0002 (canal X) carregado com sucesso.
📊 Sinal recebido: 120000 amostras
✅ Sinal limpo: 120000 amostras
   Duração da época: 120s (24000 amostras)
   Número de épocas possíveis: 5
📦 Épocas criadas: 5 épocas
   Nomes das épocas: [np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5')]

🔍 INICIANDO PROCESSAMENTO DAS ÉPOCAS

📌 Processando época '1'...
   Tamanho do sinal da época: 24000 amostras
   ✓ Picos R encontrados: 173
   ✓ Método 1 (nk.hrv_time) processado
   ⚠️  Método 2 falhou, usando Método 1 como fallback
   ✓ Métricas morfológicas processadas
   ✅ Época processada com sucesso!

📌 Processando época '2'...
   Tamanho do sinal da época: 24000 amostras
   ✓ Picos R encontrados: 171
   ✓ Método 1 (nk.hrv_time) processado
   ⚠️  Método 2 falhou, usando Método 1 como fallback
   ✓ Métricas morfológicas processadas
   ✅ Época processada com sucesso!

📌 Processando época '3'...
   Tamanho do sinal da época: 24000

In [9]:
teste

,Epoca,HRV_MeanNN_v1,HRV_MinNN_v1,HRV_MaxNN_v1,RR_Range_v1,HRV_SDNN_v1,HRV_RMSSD_v1,HRV_pNN50_v1,HRV_MeanNN_v2,HRV_MinNN_v2,HRV_MaxNN_v2,RR_Range_v2,Bradycardia_v2,HRV_SDNN_v2,HRV_RMSSD_v2,HRV_pNN50_v2,QRS_Duration_Mean,QT_Interval_Mean
0,1,693.895349,635.0,755.0,120.0,28.075724,11.584927,0.581395,693.895349,635.0,755.0,120.0,0,28.075724,11.584927,0.581395,78.352601,234.404829
1,2,701.058824,630.0,780.0,150.0,27.884951,10.748607,0.000000,701.058824,630.0,780.0,150.0,0,27.884951,10.748607,0.000000,84.678363,229.471171
2,3,695.906433,640.0,735.0,95.0,19.210062,6.870654,0.000000,695.906433,640.0,735.0,95.0,0,19.210062,6.870654,0.000000,78.837209,227.830072
3,4,709.613095,635.0,760.0,125.0,25.914397,8.538395,0.000000,709.613095,635.0,760.0,125.0,0,25.914397,8.538395,0.000000,79.047619,229.542326
4,5,704.112426,635.0,755.0,120.0,25.683150,8.720187,0.000000,704.112426,635.0,755.0,120.0,0,25.683150,8.720187,0.000000,78.852941,223.590157


In [92]:
teste

""


In [12]:
dados

In [ ]:
import pandas as pd

link = r'D:\cox-models-sudden-death\01_Dataset\dados_csv_info_definitions\subject-info_formatado.csv'


df = pd.read_csv(link, sep=';')


df[['Patient ID',
    'minimum RR (ms)',
    'Average RR (ms)',
    'maximum RR (ms)',
    'RR range (ms)',
    'Average RR (ms)',
    'Bradycardia',
    'SDNN (ms)',
    'SDANN (ms)',
    'RMSSD (ms)',
    'pNN50 (%)',
    'QRS duration (ms)',
    'QT interval (ms)'
]]

In [91]:
df

,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,SCD_4years SinusRhythm,HF_4years SinusRhythm,Age,Gender (male=1),Weight (kg),...,Angiotensin-II receptor blocker (yes=1),Anticoagulants/antitrombotics (yes=1),Betablockers (yes=1),Digoxin (yes=1),Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1)
0,P0001,2065,1460,NaN,0,0,0,58,1,83,...,0,1,1,1,1,0,0,0,1,0
1,P0002,2045,1460,NaN,0,0,0,58,1,74,...,1,1,1,0,0,0,1,0,0,0
2,P0003,2044,1460,NaN,0,0,0,69,1,83,...,1,1,1,1,1,0,0,0,0,0
3,P0004,2044,1460,NaN,0,0,0,56,0,84,...,1,1,1,0,1,1,0,0,0,0
4,P0005,2043,1460,NaN,0,0,0,70,1,97,...,0,1,1,0,1,0,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
987,P1064,1393,1393,NaN,0,0,0,63,0,67,...,1,1,1,0,1,0,0,0,0,0
988,P1065,1393,1393,NaN,0,0,0,80,1,79,...,0,1,0,1,1,0,0,0,1,0
989,P1066,1387,1387,NaN,0,0,0,72,0,81,...,0,0,0,0,1,0,0,0,1,0
990,P1073,1365,1365,NaN,0,0,0,70,0,63,...,0,1,1,0,1,0,0,0,1,0


In [87]:
# Métrica Desejada (do CSV Clínico)	Coluna Correspondente no NeuroKit2	Função NeuroKit2 a ser Usada
# Average RR (ms)	HRV_MeanNN	nk.hrv_time()
# minimum RR (ms)	HRV_MinNN	nk.hrv_time()
# maximum RR (ms)	HRV_MaxNN	nk.hrv_time()
# RR range (ms)	(Cálculo: HRV_MaxNN - HRV_MinNN)	nk.hrv_time()
# Bradycardia	(Cálculo: ECG_Rate_Mean < 60)	nk.hrv_time() (usa a coluna ECG_Rate_Mean)
# SDNN (ms)	HRV_SDNN	nk.hrv_time()
# RMSSD (ms)	HRV_RMSSD	nk.hrv_time()
# pNN50 (%)	HRV_pNN50	nk.hrv_time()
# QRS duration (ms)	ECG_Rate_Mean_QRS_Duration	nk.ecg_intervalrelated()
# QT interval (ms)	ECG_Rate_Mean_QT_Interval	nk.ecg_intervalrelated()




# HRV_MeanNN
# HRV_MinNN
# HRV_MaxNN
# RR range (ms)	(Cálculo: HRV_MaxNN - HRV_MinNN)
# Bradycardia
# HRV_SDNN
# HRV_pNN50

In [65]:
import neurokit2 as nk

# Download data
data = nk.data("bio_resting_5min_100hz")

# Process the data
df, info = nk.ecg_process(data["ECG"], sampling_rate=100)

# Single dataframe is passed
teste = nk.ecg_intervalrelated(df, sampling_rate=100)
 




In [ ]:
hrv_metrics2 = nk.ecg_intervalrelated(df, sampling_rate=100)

# peaks, info = nk.ecg_peaks(df, sampling_rate=100)

# hrv_metrics2 = nk.hrv(peaks, sampling_rate=100, show=False)

# ==========================================================
# ✅ NOVO: Extração de todas as métricas de HRV desejadas
# ==========================================================
hrv_mean_nnV2 = hrv_metrics2['HRV_MeanNN'].values[0]
hrv_min_nnV2 = hrv_metrics2['HRV_MinNN'].values[0]
hrv_max_nnV2 = hrv_metrics2['HRV_MaxNN'].values[0]
rr_rangeV2 = hrv_max_nnV2 - hrv_min_nnV2  # Cálculo derivado
bradycardiaV2 = int(hrv_metrics2['ECG_Rate_Mean'].values[0] < 60) # Cálculo derivado
hrv_sdnnV2 = hrv_metrics2['HRV_SDNN'].values[0]
hrv_rmssdV2 = hrv_metrics2['HRV_RMSSD'].values[0]
hrv_pnn50V2 = hrv_metrics2['HRV_pNN50'].values[0]


# # ===== Detecção de Bradicardia (frequência < 60 bpm) =====
# avg_heart_rate = 60000 / avg_rr
# bradycardia = int(avg_heart_rate < 60)

In [85]:
hrv_metrics2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 92 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   ECG_Rate_Mean                 1 non-null      object
 1   HRV_MeanNN                    1 non-null      object
 2   HRV_SDNN                      1 non-null      object
 3   HRV_SDANN1                    1 non-null      object
 4   HRV_SDNNI1                    1 non-null      object
 5   HRV_SDANN2                    1 non-null      object
 6   HRV_SDNNI2                    1 non-null      object
 7   HRV_SDANN5                    1 non-null      object
 8   HRV_SDNNI5                    1 non-null      object
 9   HRV_RMSSD                     1 non-null      object
 10  HRV_SDSD                      1 non-null      object
 11  HRV_CVNN                      1 non-null      object
 12  HRV_CVSD                      1 non-null      object
 13  HRV_MedianNN            

In [ ]:
[
    'HRV_MeanNN',
    'HRV_RMSSD',
    'HRV_SDNN',
    'HRV_SDANN',
    'HRV_pNN50',
    'ECG_Rate_Mean_QRS_Duration',
    'ECG_Rate_Mean_QT_Interval'
]


[
    'minimum RR (ms) ',
    'Average RR (ms)',
    'maximum RR (ms)',
    'RR range (ms)',
    'Average RR (ms)',
    'Bradycardia',
    'SDNN (ms)',
    'SDANN (ms)',
    'RMSSD (ms)',
    'pNN50 (%)'
    'QRS duration (ms)',
    'QT interval (ms)'
]



In [ ]:
eletrocardiogramas_holter = [
    'Hig-resolution ECG available',
    'ECG rhythm ',


    'QRS duration (ms)',
    'QT interval (ms)',

]


holter = [
    'Holter available',
    'Holter onset (hh:mm:ss)',
    'Holter  rhythm ',
    'minimum RR (ms) ',
    'Average RR (ms)',
    'maximum RR (ms)',
    'RR range (ms)',
    'Number of ventricular premature beats in 24h',
    'Extrasystole couplets ',
    'Ventricular Extrasystole',
    'Non-sustained ventricular tachycardia (CH>10)',
    'Longest RR pause (ms)',
    'Bradycardia',
    'SDNN (ms)',
    'SDANN (ms)',
    'RMSSD (ms)',
    'pNN50 (%)'
]

Index(['Patient ID', 'Follow-up period from enrollment (days)', 'days_4years',
       'Exit of the study', 'Cause of death', 'SCD_4years SinusRhythm',
       'HF_4years SinusRhythm', 'Age', 'Gender (male=1)', 'Weight (kg)',
       ...
       'Angiotensin-II receptor blocker (yes=1)',
       'Anticoagulants/antitrombotics  (yes=1)', 'Betablockers (yes=1)',
       'Digoxin (yes=1)', 'Loop diuretics (yes=1)', 'Spironolactone (yes=1)',
       'Statins (yes=1)', 'Hidralazina (yes=1)', 'ACE inhibitor (yes=1)',
       'Nitrovasodilator (yes=1)'],
      dtype='object', length=105)

In [ ]:
df['HRV_MeanNN']


'HRV_MeanNN': hrv['HRV_MeanNN'].values[0],
'HRV_RMSSD': hrv['HRV_RMSSD'].values[0],
'QRS_Duration_Mean': qrs['ECG_Rate_Mean_QRS_Duration'].values[0],
'QT_Interval_Mean': qrs['ECG_Rate_Mean_QT_Interval'].values[0] 

KeyError: 'HRV_MeanNN'